# CSE 151B Competition Notebook — vLLM Backend

Works on **Google Colab (T4)**, **Kaggle (T4/P100)**, and **UCSD DataHub** with a GPU runtime.

Pipeline:
1. Install dependencies
2. Load dataset
3. Run inference via vLLM
4. Save submission CSV

The single entry point required by the competition is `run_inference()` in Section 5.

## 1. Environment Setup

**Run this cell once per session.** vLLM takes 5–10 minutes to install.  
After installation, **restart the runtime** (Runtime → Restart runtime on Colab, or Kernel → Restart on Kaggle/DataHub) before continuing.

> On Colab: Runtime → Change runtime type → T4 GPU (before running anything)

In [ ]:
# ── Install dependencies (run once, then restart runtime) ─────────────────────
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

# vLLM — pinned to a stable release that supports Qwen3 and T4/A100
# 0.8.5 is the latest stable as of May 2026 and has full Qwen3 support
pip("vllm==0.8.5")

# Other deps
pip("sympy", "numpy", "tqdm", "huggingface_hub")

print("\n✓ Installation complete — RESTART THE RUNTIME NOW before continuing.")

## 2. Verify GPU & vLLM

Run this after restarting the runtime. Confirms CUDA is visible and vLLM imports correctly.

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU detected. On Colab: Runtime → Change runtime type → T4 GPU. "
    "On Kaggle: Settings → Accelerator → GPU."
)

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU  : {gpu_name}")
print(f"VRAM : {vram_gb:.1f} GB")

# Verify vLLM import — this is the most common failure point
try:
    from vllm import LLM, SamplingParams
    print("vLLM : imported successfully")
except ImportError as e:
    print(f"vLLM import failed: {e}")
    print("Did you restart the runtime after installation?")

## 3. HuggingFace Login

Qwen3-4B is a gated model — you need a HuggingFace account and must accept the model license at  
https://huggingface.co/Qwen/Qwen3-4B

**On Colab**: add your token as a Secret named `HF_TOKEN` (key icon in the left sidebar), then run the cell below.  
**On Kaggle**: add it under Add-ons → Secrets.  
**On DataHub**: paste it directly (don't commit to git).

In [ ]:
import os

# ── Try reading from Colab/Kaggle secrets first ───────────────────────────────
HF_TOKEN = None

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    print("Token loaded from Colab secrets.")
except Exception:
    pass

if HF_TOKEN is None:
    # Kaggle secret
    HF_TOKEN = os.environ.get("HF_TOKEN")
    if HF_TOKEN:
        print("Token loaded from environment variable.")

if HF_TOKEN is None:
    # Fallback: paste directly (do NOT commit this to git)
    HF_TOKEN = ""  # ← paste your token here if not using secrets

assert HF_TOKEN, "HF_TOKEN is empty. Add it as a Colab/Kaggle secret or paste it above."

from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
print("HuggingFace login successful.")

## 4. Configuration

All hyperparameters are set here — these are the **final values used for submission**.

In [ ]:
import json, csv, re, os
from pathlib import Path
from typing import Optional
from tqdm import tqdm

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_PATH   = "data/private.jsonl"   # private test set (no answers)
OUTPUT_PATH = "results/submission.csv"

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_NAME  = "Qwen/Qwen3-4B"        # HuggingFace model ID

# ── Inference hyperparameters (final values for reproducibility) ───────────────
MAX_TOKENS  = 32768
TEMPERATURE = 0.2
TOP_P       = 0.85
TOP_K       = 20

# ── vLLM engine settings ──────────────────────────────────────────────────────
# gpu_memory_utilization: fraction of VRAM to give vLLM
# 0.90 is safe for T4 (16GB) and leaves headroom for CUDA overhead
GPU_MEM_UTIL = 0.90

# max_model_len: caps the KV cache. 32768 matches MAX_TOKENS.
# Lower this to 16384 if you get CUDA OOM on a 16GB GPU.
MAX_MODEL_LEN = 32768

print(f"Model       : {MODEL_NAME}")
print(f"Data        : {DATA_PATH}")
print(f"Output      : {OUTPUT_PATH}")
print(f"Max tokens  : {MAX_TOKENS}")
print(f"Temperature : {TEMPERATURE}")

## 5. Prompts & Answer Extraction

In [ ]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step.\n\n"

    "\u2550\u2550\u2550 FINAL ANSWER FORMAT \u2014 THE MOST IMPORTANT RULE \u2550\u2550\u2550\n"
    "At the very LAST LINE of your response, place ALL answers inside exactly ONE \\boxed{}.\n"
    "  DO:     \\boxed{380, 315, 13, 310}  (all parts, comma-separated, one box)\n"
    "  DO:     \\boxed{5/8}  (single answer)\n"
    "  DON'T:  box each sub-answer in a separate \\boxed{} throughout the solution\n"
    "  DON'T:  \\boxed{380}  ...text...  \\boxed{315}  ...text...  \\boxed{13}\n"
    "Even if you use \\boxed{} for intermediate steps during working, you MUST finish with "
    "a single combined \\boxed{a, b, c} on the very last line \u2014 all answers, in the order asked.\n"
    "Never leave \\boxed{} empty.\n\n"

    "\u2550\u2550\u2550 EXACT FORM RULES \u2550\u2550\u2550\n"
    "1. SYMBOLIC OVER NUMERIC \u2014 if the answer is a function applied to given constants, "
    "write the expression, NOT a decimal:\n"
    "  DO:     \\arctan(4.76)          DON'T: 1.3635\n"
    "  DO:     \\ln(0.5)/\\ln(0.96584)  DON'T: 19.94\n"
    "  DO:     (1/2)^{(1999-1963)/31} DON'T: 0.447\n"
    "2. DECIMAL PRECISION \u2014 when a decimal is required, give at minimum 6 significant digits:\n"
    "  DO:     7.79744   DON'T: 7.80  |  DO: 442.857   DON'T: 442.86  |  DO: 12.0814  DON'T: 12.08\n"
    "3. PRESERVE STRUCTURE \u2014 if the problem writes 2*8*x, write 2*8*x not 16x.\n"
    "4. EXPLICIT MULTIPLICATION \u2014 write 3*t*(1-t)^2, not 3t(1-t)^2.\n"
    "5. EXPONENTIALS \u2014 write \\exp(0.016*t) or e^{0.016t}, not standalone e^0.016t.\n"
    "6. FRACTIONS \u2014 use exact fractions (5/8) for rational results.\n"
    "7. ORDER \u2014 answer multi-part questions in the exact order the problem asks.\n"
    "8. NO ANGLE BRACKETS \u2014 do not wrap answers in <> brackets.\n\n"

    "Before writing the final \\boxed{}, verify your answer satisfies the original problem. "
    "Commit to your best answer."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices carefully, then select the single best answer.\n\n"
    "STEP 1 \u2014 Solve: Work through the problem step-by-step to derive your answer.\n"
    "STEP 2 \u2014 Match: Compare your result against every option:\n"
    "  a) Check algebraic/symbolic equivalence (e.g. pi*sqrt(a) = pi*a^{1/2}, "
    "4/3*ln(3) = 2/3*ln(9), 1-cos^2(x) = sin^2(x)).\n"
    "  b) If options look different, plug in a concrete numeric value for any free variable "
    "and evaluate BOTH your answer and each option \u2014 pick the one whose value matches yours.\n"
    "  c) If two options appear numerically equal, prefer the one whose algebraic form "
    "matches your derivation most directly.\n"
    "STEP 3 \u2014 Commit: Trust your derivation. Do not abandon a correct answer just because "
    "the option looks different in form.\n\n"
    "You MUST always pick one of the given letters, never say none match. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)


def build_messages(question: str, options: Optional[list]) -> list[dict]:
    """Return a messages list in OpenAI chat format for vLLM."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        user_content = f"{question}\n\nOptions:\n{opts_text}"
        system = SYSTEM_PROMPT_MCQ
    else:
        user_content = question
        system = SYSTEM_PROMPT_MATH
    return [
        {"role": "system", "content": system},
        {"role": "user",   "content": user_content},
    ]


def extract_boxed(text: str) -> str:
    """Extract content of the LAST \\boxed{} in text."""
    # Handles nested braces correctly
    matches = []
    i = 0
    while i < len(text):
        idx = text.find(r'\boxed{', i)
        if idx == -1:
            break
        # Find matching closing brace
        depth = 0
        j = idx + len(r'\boxed{')
        start = j
        while j < len(text):
            if text[j] == '{':
                depth += 1
            elif text[j] == '}':
                if depth == 0:
                    matches.append(text[start:j])
                    break
                depth -= 1
            j += 1
        i = idx + 1
    return matches[-1].strip() if matches else ""


print("Prompts and extraction utilities loaded.")

## 6. `run_inference()` — Competition Entry Point

This is the single function required by the competition spec.  
Call `run_inference()` to reproduce the full pipeline end-to-end.

In [ ]:
def run_inference(
    data_path:   str = DATA_PATH,
    output_path: str = OUTPUT_PATH,
    model_name:  str = MODEL_NAME,
) -> str:
    """
    Full end-to-end inference pipeline.

    Parameters
    ----------
    data_path   : path to the .jsonl dataset (private test set)
    output_path : path to write the submission CSV
    model_name  : HuggingFace model ID or local path

    Returns
    -------
    output_path : path to the written CSV
    """
    import torch
    from vllm import LLM, SamplingParams

    # ── 1. Load dataset ───────────────────────────────────────────────────────
    print("Loading dataset...")
    data = [json.loads(line) for line in open(data_path, encoding="utf-8")]
    n_mcq  = sum(bool(d.get("options")) for d in data)
    n_free = len(data) - n_mcq
    print(f"Loaded {len(data)} questions ({n_mcq} MCQ, {n_free} free-form)")

    # ── 2. Resume: skip already-completed questions ───────────────────────────
    out_path = Path(output_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    done: dict[int, str] = {}  # id -> response
    if out_path.exists():
        with open(out_path, newline="", encoding="utf-8") as f:
            for row in csv.DictReader(f):
                resp = row["response"]
                if resp and not resp.startswith("ERROR:"):
                    done[int(row["id"])] = resp
        print(f"Resuming: {len(done)} done, {len(data) - len(done)} remaining")

    remaining = [d for d in data if d["id"] not in done]
    if not remaining:
        print("All questions already completed. Writing CSV.")
        _write_csv(out_path, data, done)
        return str(out_path)

    # ── 3. Load vLLM engine ───────────────────────────────────────────────────
    print(f"\nLoading model: {model_name}")
    print("(This takes ~2–4 minutes on first load)")

    llm = LLM(
        model=model_name,
        dtype="float16",             # T4 / P100 don't support bfloat16
        gpu_memory_utilization=GPU_MEM_UTIL,
        max_model_len=MAX_MODEL_LEN,
        # enable_thinking=True is handled via the chat template for Qwen3
        # No extra flag needed — Qwen3's tokenizer enables thinking by default
        trust_remote_code=True,
        enforce_eager=False,         # keep CUDA graphs enabled for speed
    )
    print("Model loaded.")

    sampling_params = SamplingParams(
        temperature=TEMPERATURE,
        top_p=TOP_P,
        top_k=TOP_K,
        max_tokens=MAX_TOKENS,
    )

    # ── 4. Build prompts using the tokenizer's chat template ──────────────────
    # Using apply_chat_template is required for Qwen3 thinking mode to work
    # correctly. vLLM's tokenizer object exposes this.
    tokenizer = llm.get_tokenizer()

    prompts = []
    for item in remaining:
        messages = build_messages(item["question"], item.get("options"))
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            # Qwen3 thinking mode: enabled by default when the template sees
            # no explicit /think token — leave enable_thinking out here
        )
        prompts.append(prompt)

    # ── 5. Batch inference ────────────────────────────────────────────────────
    print(f"\nRunning inference on {len(remaining)} questions...")
    # vLLM processes all prompts in one call — it handles batching internally
    outputs = llm.generate(prompts, sampling_params)

    # ── 6. Collect results ────────────────────────────────────────────────────
    for item, output in zip(remaining, outputs):
        response = output.outputs[0].text
        done[item["id"]] = response

    # ── 7. Write CSV ──────────────────────────────────────────────────────────
    _write_csv(out_path, data, done)

    n_empty = sum(1 for d in data if not done.get(d["id"], "").strip())
    print(f"\nDone. {len(data)} questions processed, {n_empty} empty responses.")
    print(f"Submission saved to: {out_path}")
    return str(out_path)


def _write_csv(out_path: Path, data: list, done: dict) -> None:
    """Write id,response CSV in original dataset order."""
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["id", "response"])
        writer.writeheader()
        for item in data:
            writer.writerow({
                "id":       item["id"],
                "response": done.get(item["id"], ""),
            })


print("run_inference() defined and ready.")

## 7. Run

**Full run** (all questions):
```python
run_inference()
```

**Quick test** on a subset (to verify the pipeline before committing to the full run):
```python
run_inference(data_path="data/private.jsonl")  # limit handled below
```

Set `TEST_MODE = True` to run on 20 questions only.

In [ ]:
TEST_MODE = True   # ← set False for full run

if TEST_MODE:
    # Write a 20-question subset to a temp file and run on that
    import tempfile
    data_all = [json.loads(line) for line in open(DATA_PATH, encoding="utf-8")]
    subset   = data_all[:20]
    tmp      = tempfile.NamedTemporaryFile(mode="w", suffix=".jsonl",
                                           delete=False, encoding="utf-8")
    for item in subset:
        tmp.write(json.dumps(item) + "\n")
    tmp.close()
    print(f"TEST MODE: running on 20 questions → {tmp.name}")
    run_inference(data_path=tmp.name, output_path="results/test_run.csv")
else:
    run_inference()

## 8. Score Results (Public Set Only)

Only run this section if you have the **public** dataset with ground-truth answers.  
Skip for the private test set submission.

Replace `RESULTS_PATH` and `PUBLIC_DATA_PATH` with your actual paths.

In [ ]:
RESULTS_PATH     = "results/submission.csv"  # your generated CSV
PUBLIC_DATA_PATH = "data/public.jsonl"       # public set WITH answers

# Load ground truth
gt = {}
for line in open(PUBLIC_DATA_PATH, encoding="utf-8"):
    item = json.loads(line)
    gt[item["id"]] = item

# Load predictions
preds = {}
with open(RESULTS_PATH, newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        preds[int(row["id"])] = row["response"]

# MCQ scoring
def extract_letter(text: str) -> str:
    boxed = extract_boxed(text)
    if boxed and len(boxed) == 1 and boxed.isalpha():
        return boxed.upper()
    # Fallback: last standalone capital letter
    matches = re.findall(r'\b([A-Z])\b', text.upper())
    return matches[-1] if matches else ""

# Free-form scoring via Judger
import sys
sys.path.insert(0, ".")
try:
    from judger import Judger
    judger = Judger(strict_extract=False)
    has_judger = True
except ImportError:
    print("WARNING: judger.py not found — free-form scoring will be skipped.")
    has_judger = False

mcq_correct = mcq_total = free_correct = free_total = 0

for qid, item in gt.items():
    if qid not in preds:
        continue
    response = preds[qid]
    gold     = item["answer"]
    is_mcq   = bool(item.get("options"))

    if is_mcq:
        mcq_total += 1
        if extract_letter(response) == gold.strip().upper():
            mcq_correct += 1
    else:
        free_total += 1
        if has_judger:
            gold_list = gold if isinstance(gold, list) else [gold]
            if judger.auto_judge(response, gold_list, options=[[]] * len(gold_list)):
                free_correct += 1

def pct(n, d):
    return f"{n/d*100:.2f}%" if d else "N/A"

total_correct = mcq_correct + free_correct
total         = mcq_total + free_total

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {mcq_correct:4d} / {mcq_total:4d}  ({pct(mcq_correct, mcq_total)})")
print(f"  Free-form  : {free_correct:4d} / {free_total:4d}  ({pct(free_correct, free_total)})")
print(f"  Overall    : {total_correct:4d} / {total:4d}  ({pct(total_correct, total)})")
print("=" * 50)

## 9. README Template

Copy the output below into your `README.md` before submitting to Gradescope.

In [ ]:
readme = f"""
# CSE 151B Spring 2026 — Math Reasoning Competition

## Hardware
- GPU: [fill in — e.g. NVIDIA T4 16GB on Kaggle]
- Approximate inference time: [fill in — e.g. ~3 hours for 943 questions]

## Model
- Base model: `{MODEL_NAME}` (no fine-tuning)
- Inference: vLLM with float16, thinking mode enabled via Qwen3 chat template

## Setup

1. Install dependencies (restart runtime after):
```
pip install vllm==0.8.5 sympy numpy tqdm huggingface_hub
```

2. Place the private dataset at `data/private.jsonl`.

3. Set your HuggingFace token (accept model license at https://huggingface.co/Qwen/Qwen3-4B first):
```
export HF_TOKEN=your_token_here
```

## Reproduce Results

```python
from competition_notebook import run_inference
run_inference()  # writes results/submission.csv
```

Or run the notebook top-to-bottom with TEST_MODE = False in Section 7.

## Hyperparameters
- temperature: {TEMPERATURE}
- top_p:       {TOP_P}
- top_k:       {TOP_K}
- max_tokens:  {MAX_TOKENS}
"""
print(readme)